### 1.问题
只有bash时，所有操作都走shell。cat截断不可预测，sed遇到特殊字符就崩，每次bash调用都是不受约束的安全面。专用工具（read_file，write_file）可以在工具层面做路径沙箱。
关键步骤：做到加工具不需要改循环
### 2.解决工作原理
1. 每个工具都有一个处理函数。路径沙箱防止逃逸出工作区。
```python
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    # 进行其他安全检查
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_read(path: str, limit: int = None) -> str:
    text = safe_path(path).read_text()
    lines = text.splitlines()
    if limit and limit < len(lines):
        lines = lines[:limit]
    return "\n".join(lines)[:50000]
```
2. dispatch map将工具名映射到处理函数。
```python
TOOL_HANDLERS = {
    "bash":    lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"], kw["limit"]),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}
```
3. 循环中按名称查找处理函数。循环体本身和01完全一致。
```python
for tool_call in message.tool_calls:
    handler = TOOL_HANDLERS.get(tool_call.function.name)
    output = handler(**tool_call.function.arguments) if handler else f"Unknown tool: {tool_call.function.name}"
    tool_results.append({
        "tool_call_id": tool_call.id,
        "role": "tool",
        "name": "bash",
        "content": output,
    })
```
加工具 = 加 handler + 加 schema。循环永远不变。

In [8]:
import os
import subprocess
from pathlib import Path
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url=os.getenv("OPENAI_API_BASE"))

WORKDIR = Path.cwd()

SYSTEM_PROMPT = f"You are a coding agent at {WORKDIR}. Use tools to solve tasks. Act, do not explain."

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"

def run_read(path: str, limit: int = None) -> str:
    try:
        text = safe_path(path).read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"...({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: {old_text} not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

TOOL_HANDLERS = {
    "bash": lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"]),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": "Run a shell command.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "the shell command to execute"
                    }
                },
                "required": ["command"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read file contents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "the path of the file",
                    },
                    "limit": {
                        "type": "int",
                        "description": "the limit of the file content.",
                    }
                },
                "required": ["path"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "write content to file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "file path",
                    },
                    "content": {
                        "type": "string",
                        "description": "the content to write to the file.",
                    }
                },
                "required": ["path", "content"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": "Replace exact text in file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "file path",
                    },
                    "old_text": {
                        "type": "string",
                        "description": "the text to be replaced",
                    },
                    "new_text": {
                        "type": "string",
                        "description": "the text to replace the old text",
                    }
                },
                "required": ["path", "old_text", "new_text"],
            },
        },
    }
]

def agent_loop(messages: list):
    while True:
        responese = client.chat.completions.create(
            model="Pro/MiniMaxAI/MiniMax-M2.5",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        message = responese.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            if message.content:
                print(message.content)
            return
        
        # 处理工具调用
        tool_results = []
        for tool_call in message.tool_calls:
            handler = TOOL_HANDLERS.get(tool_call.function.name)
            output = handler(**json.loads(tool_call.function.arguments)) if handler else f"Unknown tool: {tool_call.function.name}"
            print(f">{tool_call.function.name}: {output[:200]}")
            tool_results.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "content": output,
            })
        
        messages.extend(tool_results)

if __name__ == "__main__":
    history = []
    while True:
        try:
            query = input("请输入: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({"role": "user", "content": query})
        agent_loop(history)
        response_content = history[-1].content
        if isinstance(response_content, list):
            for block in response_content:
                if hasattr(block, "text"):
                    print(block.text)
        else:
            print(response_content)
        print()
       


>bash: 'ls' 不是内部或外部命令，也不是可运行的程序
或批处理文件。
>bash: 驱动器 D 中的卷是 Data
 卷的序列号是 3411-3319

 d:\workspace\ai\langchain_learn\learn_claude_code 的目录

2026/03/22  10:44             3,446 00_agent开发是什么.ipynb
               1 个文件          3,446 字节
              
>read_file: {
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a7c809a9",
   "metadata": {},
   "source": [
    "### 1.Agent是什么\n",
    "Agent是模型，不是框架，不是提示词链，不是拖拽式工作流。  \n",
    "Agent的本质是一个学会了行动的模型，从来不是外面那层


这是一个关于 **Agent 开发** 的 Jupyter Notebook 文件，我来帮你整理一下内容：

---

## 文件内容概要

### 1. Agent 是什么？
- Agent 是**模型**，不是框架、不是提示词链、也不是拖拽式工作流
- Agent 的本质是一个**学会了行动的模型**

### 2. 心智转换：从开发 Agent 到开发 Harness

当说"开发 Agent"时，有两种可能：
1. **训练模型**：通过强化学习、微调、RLHF等方法调整模型权重
2. **构建 Harness**：编写代码为模型提供可操作的环境（这是大多数人在做的事）

**Harness = Tools + Knowledge + Observation + Action Interfaces + Permissions**
- 🔧 **Tools**: 文件读写、Shell、网络、数据库、浏览器
- 📚 **Knowledge**: 产品文档、领域资料、API 规范、风格指南
- 👁️ **Observation**: git diff、错误日志、浏览器状态
- 🎯 **Action**: CLI命令、API调用、UI交互
- 🛡️ **Per